In [1]:
import pandas as pd

df = pd.read_csv("../data/telco_churn.csv")

In [2]:
# Supprimer customerID
df = df.drop(columns=['customerID'])

In [4]:
# Vérifier TotalCharges
print(df['TotalCharges'].dtype)

str


In [8]:
# Vérifier les valeurs problématiques
(df['TotalCharges'] == ' ').sum()

np.int64(11)

In [11]:
# Conversion
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [15]:
print(df['TotalCharges'].dtype)

float64


In [13]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [16]:
# Traitement des NaN
df = df.dropna()

In [17]:
print(df.shape)

(7032, 20)


In [6]:
# Encoder la cible
df['Churn'] = df["Churn"].map({
    'No': 0,
    'Yes': 1
})

In [18]:
# Séparer X et y
X = df.drop('Churn', axis=1)

y = df['Churn']

In [19]:
# Identifier les colonnes catégorielles
cat_cols = X.select_dtypes(include='object').columns

C:\Users\HP\AppData\Local\Temp\ipykernel_7024\3623664290.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


In [22]:
cat_cols

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [23]:
# One-Hot Encoding
X = pd.get_dummies(
    X,
    drop_first=True
)

In [24]:
print(X.shape)

print(y.shape)

print(X.head())

(7032, 30)
(7032,)
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0              0       1           29.85         29.85        False   
1              0      34           56.95       1889.50         True   
2              0       2           53.85        108.15         True   
3              0      45           42.30       1840.75         True   
4              0       2           70.70        151.65        False   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0         True           False             False   
1        False           False              True   
2        False           False              True   
3        False           False             False   
4        False           False              True   

   MultipleLines_No phone service  MultipleLines_Yes  ...  \
0                            True              False  ...   
1                           False              False  ...   
2                           False              False  ... 

In [30]:
# Séparation Train / Test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(5625, 30)
(1407, 30)


In [26]:
# Baseline Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr.fit(X_train, y_train)

C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=2000, random_state=42)

In [33]:
# Évaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))

print("ROC AUC :", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.58      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

ROC AUC : 0.8360532895724513


In [28]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [32]:
# Étape 5 : Évaluation Random Forest
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print(classification_report(
    y_test,
    y_pred_rf
))

print(
    "ROC AUC RF:",
    roc_auc_score(
        y_test,
        y_proba_rf
    )
)

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC AUC RF: 0.8185001889517577


In [34]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="importance",
    ascending=False
)

print(importance.head(15))

                           feature  importance
3                     TotalCharges    0.193238
1                           tenure    0.170365
2                   MonthlyCharges    0.168902
10     InternetService_Fiber optic    0.038595
28  PaymentMethod_Electronic check    0.037837
25               Contract_Two year    0.032238
4                      gender_Male    0.028753
13              OnlineSecurity_Yes    0.027205
26            PaperlessBilling_Yes    0.025457
5                      Partner_Yes    0.023462
24               Contract_One year    0.022874
19                 TechSupport_Yes    0.022659
15                OnlineBackup_Yes    0.022333
0                    SeniorCitizen    0.021001
6                   Dependents_Yes    0.020104


In [36]:
# Sauvegarde du modèle (model.pkl + features.pkl)

import joblib

features = X.columns.tolist()

# save model
joblib.dump(rf, "model.pkl")

# save features
joblib.dump(features, "features.pkl")

print("Model & features saved successfully")

Model & features saved successfully


In [37]:
for f in features:
    print(f)

SeniorCitizen
tenure
MonthlyCharges
TotalCharges
gender_Male
Partner_Yes
Dependents_Yes
PhoneService_Yes
MultipleLines_No phone service
MultipleLines_Yes
InternetService_Fiber optic
InternetService_No
OnlineSecurity_No internet service
OnlineSecurity_Yes
OnlineBackup_No internet service
OnlineBackup_Yes
DeviceProtection_No internet service
DeviceProtection_Yes
TechSupport_No internet service
TechSupport_Yes
StreamingTV_No internet service
StreamingTV_Yes
StreamingMovies_No internet service
StreamingMovies_Yes
Contract_One year
Contract_Two year
PaperlessBilling_Yes
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
PaymentMethod_Mailed check
